# Landslide Training on Google Colab

This notebook trains the repository's real RGB landslide-segmentation models on the **CAS Landslide Dataset** with event-disjoint train/validation/test roles. It uses the project data loaders, preprocessing, structural features, losses, metrics, trainer, checkpoint format, and deployment tools rather than duplicating them in notebook cells.

The workflow does not generate synthetic scientific results and does not contain precomputed metrics. CAS is CC BY-NC 4.0 with source-specific imagery terms; review those terms before downloading or redistributing data or weights. HR-GLDD evaluation is optional and uses RGB only, without fine-tuning.


## Before running

1. In Colab, select **Runtime > Change runtime type > GPU**. A Tesla T4 is suitable, but Colab chooses the accelerator.
2. Configure only the paths and run controls in the next cell. Use `CAS_MODE = "mounted"` when CAS is already extracted on Drive. Use `CAS_MODE = "download"` for checksum-verified, one-time downloads of all official archives (about 9.19 GB compressed, plus extracted data).
3. Keep `RUN_NAME` stable when resuming. `last.pt` is saved atomically to Drive after every epoch.
4. The complete nine-event split policy is mandatory. A Wenchuan-only subset is useful for plumbing checks but is not a valid scientific train/test experiment.


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/asrith413/landslide-detection.git"
REPO_REVISION = "main"  # Pin a full commit SHA for a final experiment.
COLAB_REPO = Path("/content/landslide-detection")

DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/landslide-colab")
CAS_MODE = "mounted"  # "mounted" or "download"
CAS_ARCHIVE_ROOT = DRIVE_PROJECT_ROOT / "datasets/cas/archives"
CAS_DATA_ROOT = DRIVE_PROJECT_ROOT / "datasets/cas/extracted"
CAS_MANIFEST_PATH = DRIVE_PROJECT_ROOT / "datasets/cas/cas_event_manifest.csv"
REBUILD_CAS_MANIFEST = False
FULL_DATA_VALIDATION = True  # First run decodes all pairs; later runs reuse a hash-keyed cache.

RUN_NAME = "cas-event-v1"
EXPERIMENT_PRESETS = [
    "segformer_b0_cas",
    "lraspp_mobilenet_v3_cas",
    "csg_segformer_b0_cas",
]
EPOCHS = 30
NUM_WORKERS = 2
BATCH_SIZE_OVERRIDES = {}  # Example: {"csg_segformer_b0_cas": 1}
RUN_PROJECT_TESTS = True

HR_GLDD_ROOT = DRIVE_PROJECT_ROOT / "datasets/hr-gldd"
DOWNLOAD_HR_GLDD_TEST = False
RUN_HR_GLDD_EVALUATION = False

RUN_ONNX_EXPORT = True
RUN_QUANTIZATION = True
RUN_BENCHMARKS = True
BENCHMARK_REPETITIONS = 30


## 1. GPU and CUDA preflight


In [ ]:
import platform
import shutil
import subprocess
import sys
import torch

print("Python:", sys.version)
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("PyTorch CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required. Select a GPU runtime and reconnect.")
gpu = torch.cuda.get_device_properties(0)
free_bytes, total_bytes = torch.cuda.mem_get_info(0)
print(f"GPU: {gpu.name}; compute capability {gpu.major}.{gpu.minor}")
print(f"VRAM: {free_bytes / 2**30:.2f} GiB free / {total_bytes / 2**30:.2f} GiB total")
if "T4" not in gpu.name.upper():
    print("Note: this is not a Tesla T4. The workflow remains valid; tune batch size for available VRAM.")


## 2. Mount persistent Google Drive storage


In [ ]:
try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError("This notebook is intended to run in Google Colab.") from exc

drive.mount("/content/drive", force_remount=False)
for path in (DRIVE_PROJECT_ROOT, CAS_ARCHIVE_ROOT, CAS_DATA_ROOT, HR_GLDD_ROOT):
    path.mkdir(parents=True, exist_ok=True)
usage = shutil.disk_usage(DRIVE_PROJECT_ROOT)
print(f"Drive free space: {usage.free / 2**30:.2f} GiB")


## 3. Clone the project and install Colab dependencies


In [ ]:
import os

def run(command, cwd=None):
    print("+", " ".join(map(str, command)))
    return subprocess.run(command, cwd=cwd, check=True, text=True)

if (COLAB_REPO / ".git").is_dir():
    run(["git", "fetch", "origin", "--tags", "--prune"], COLAB_REPO)
else:
    run(["git", "clone", REPO_URL, str(COLAB_REPO)])
run(["git", "checkout", REPO_REVISION], COLAB_REPO)
if REPO_REVISION in {"main", "master"}:
    run(["git", "pull", "--ff-only", "origin", REPO_REVISION], COLAB_REPO)
GIT_SHA = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=COLAB_REPO, text=True).strip()
print("Resolved revision:", GIT_SHA)


In [ ]:
cuda_before = (torch.__version__, torch.version.cuda, torch.cuda.is_available())
run([sys.executable, "-m", "pip", "install", "-q", "-r", str(COLAB_REPO / "requirements-colab.txt")])
run([sys.executable, "-m", "pip", "install", "-q", "-e", str(COLAB_REPO), "--no-deps"] )

import importlib.metadata as metadata
import torch as installed_torch
if not installed_torch.cuda.is_available():
    raise RuntimeError(f"Dependency installation replaced or broke CUDA PyTorch (before={cuda_before}). Restart the GPU runtime.")
for package in ("torch", "torchvision", "transformers", "onnx", "onnxruntime", "numpy", "Pillow"):
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: not installed")

from landslide.deployment import collect_runtime_report
from landslide.utils import save_json
RUNTIME_REPORT = collect_runtime_report(require_cuda=True)
RUN_ROOT = DRIVE_PROJECT_ROOT / "runs" / RUN_NAME
RUN_ROOT.mkdir(parents=True, exist_ok=True)
save_json({"git_sha": GIT_SHA, "runtime": RUNTIME_REPORT}, RUN_ROOT / "runtime_preflight.json")


In [ ]:
if RUN_PROJECT_TESTS:
    run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], COLAB_REPO)
else:
    print("Project tests skipped by configuration.")


## 4. Acquire CAS once and build the event-disjoint manifest


In [ ]:
import json

from landslide.research_data import build_cas_event_manifest
from landslide.storage import download_once, extract_zip_once

CAS_LEDGER_PATH = COLAB_REPO / "ml/configs/cas_colab.json"
CAS_LEDGER = json.loads(CAS_LEDGER_PATH.read_text(encoding="utf-8"))
acquisition = []
if CAS_MODE == "download":
    for item in CAS_LEDGER["archives"]:
        archive_path = CAS_ARCHIVE_ROOT / f"{item['directory']}.zip"
        acquisition.append(download_once(
            item["url"], archive_path, checksum=f"md5:{item['md5']}",
            expected_bytes=item["bytes"], min_free_bytes=2 * 2**30,
        ))
        acquisition.append(extract_zip_once(
            archive_path, CAS_DATA_ROOT / item["directory"],
            checksum=f"md5:{item['md5']}", min_free_bytes=2 * 2**30,
        ))
elif CAS_MODE != "mounted":
    raise ValueError("CAS_MODE must be 'mounted' or 'download'.")
if not CAS_DATA_ROOT.is_dir():
    raise FileNotFoundError(f"CAS data root not found: {CAS_DATA_ROOT}")

if REBUILD_CAS_MANIFEST or not CAS_MANIFEST_PATH.is_file():
    manifest_build = build_cas_event_manifest(
        CAS_DATA_ROOT, CAS_LEDGER_PATH, CAS_MANIFEST_PATH,
        relative_paths=True, validate=True, full_decode=False,
    )
    print(json.dumps(manifest_build, indent=2))
else:
    print("Using existing manifest:", CAS_MANIFEST_PATH)
save_json(acquisition, RUN_ROOT / "cas_acquisition.json")


## 5. Validate files, masks, and event isolation


In [ ]:
from landslide.research_data import validate_event_manifest

VALIDATION_CACHE = DRIVE_PROJECT_ROOT / "datasets/cas/manifest-validation-cache.json"
MANIFEST_REPORT = validate_event_manifest(
    CAS_MANIFEST_PATH, CAS_DATA_ROOT, split_policy=CAS_LEDGER_PATH,
    require_policy_coverage=True, required_splits=("train", "val", "test"),
    full_decode=FULL_DATA_VALIDATION, expected_image_size=512,
    cache_path=VALIDATION_CACHE,
)
save_json(MANIFEST_REPORT, RUN_ROOT / "cas_manifest_validation.json")
print(json.dumps(MANIFEST_REPORT, indent=2))


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from landslide.features import IMAGENET_MEAN, IMAGENET_STD
from landslide.research_data import build_research_dataloader

DATA_CONFIG = {
    "type": "event_manifest", "manifest": str(CAS_MANIFEST_PATH),
    "data_root": str(CAS_DATA_ROOT), "split_policy": str(CAS_LEDGER_PATH),
    "require_policy_coverage": True, "required_splits": ["train", "val", "test"],
    "expected_image_size": 512, "image_size": 512,
    "full_validation": FULL_DATA_VALIDATION, "validation_cache": str(VALIDATION_CACHE),
    "pin_memory": True,
}
preview = next(iter(build_research_dataloader(DATA_CONFIG, "train", batch_size=1, augment=False, shuffle=False)))
image = preview["image"][0].permute(1, 2, 0).numpy()
image = np.clip(image * np.asarray(IMAGENET_STD) + np.asarray(IMAGENET_MEAN), 0, 1)
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1); plt.imshow(image); plt.title(preview["event"][0]); plt.axis("off")
plt.subplot(1, 2, 2); plt.imshow(preview["mask"][0, 0], cmap="gray"); plt.title("Binary mask"); plt.axis("off")
plt.tight_layout(); plt.show()


## 6. Resolve experiment configurations


In [ ]:
def resolve_experiment(name):
    source = COLAB_REPO / "ml/configs" / f"{name}.json"
    if not source.is_file():
        raise FileNotFoundError(source)
    cfg = json.loads(source.read_text(encoding="utf-8"))
    cfg["data"].update(DATA_CONFIG)
    cfg["data"]["batch_size"] = int(BATCH_SIZE_OVERRIDES.get(name, cfg["data"]["batch_size"]))
    cfg["data"]["num_workers"] = NUM_WORKERS
    cfg["train"]["epochs"] = EPOCHS
    cfg["train"]["resume"] = False  # Resume is selected only when last.pt exists.
    cfg["output"] = str(RUN_ROOT / name)
    return cfg

EXPERIMENTS = {name: resolve_experiment(name) for name in EXPERIMENT_PRESETS}
resolved_dir = RUN_ROOT / "resolved_configs"
resolved_dir.mkdir(parents=True, exist_ok=True)
for name, cfg in EXPERIMENTS.items():
    save_json(cfg, resolved_dir / f"{name}.json")
    print(name, "->", cfg["output"], "batch=", cfg["data"]["batch_size"])


In [ ]:
from landslide.model import build_model
from landslide.utils import count_parameters

sanity = preview["image"].to("cuda")
for name, cfg in EXPERIMENTS.items():
    model = build_model(cfg).to("cuda").eval()
    with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=torch.float16):
        output = model(sanity)
    print(name, f"parameters={count_parameters(model):,}", "logits=", tuple(output["logits"].shape), "embedding=", None if output["embedding"] is None else tuple(output["embedding"].shape))
    del model, output
    torch.cuda.empty_cache()


## 7. Train with AMP and Drive checkpoints

Each preset uses the same immutable event manifest and preprocessing. SegFormer-B0 and MobileNetV3/LR-ASPP are controlled baselines. CSG-SegFormer adds the existing structural extractor, compact adapters, residual gates, and training-only hard-negative region contrast. Re-running this cell resumes each interrupted experiment from its own `last.pt`.


In [ ]:
from landslide.train import run_training

TRAINING_SUMMARIES = {}
for name, cfg in EXPERIMENTS.items():
    output_dir = Path(cfg["output"])
    latest = output_dir / "last.pt"
    print(f"\n===== {name} =====")
    TRAINING_SUMMARIES[name] = run_training(
        cfg, device="cuda", resume=latest if latest.is_file() else None,
        compute_boundary=True,
    )
save_json(TRAINING_SUMMARIES, RUN_ROOT / "training_summaries.json")


In [ ]:
for name, cfg in EXPERIMENTS.items():
    history_path = Path(cfg["output"]) / "history.json"
    history = json.loads(history_path.read_text(encoding="utf-8"))
    epochs = [row["epoch"] for row in history]
    figure, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(epochs, [row["train_loss"] for row in history], label="train")
    axes[0].plot(epochs, [row["val_loss"] for row in history], label="validation")
    axes[0].set_title(f"{name}: loss"); axes[0].set_xlabel("epoch"); axes[0].legend()
    axes[1].plot(epochs, [row["val_iou"] for row in history], label="IoU")
    axes[1].plot(epochs, [row["val_dice"] for row in history], label="Dice/F1")
    axes[1].plot(epochs, [row["val_boundary_iou"] for row in history], label="Boundary IoU")
    axes[1].set_title(f"{name}: validation"); axes[1].set_xlabel("epoch"); axes[1].legend()
    figure.tight_layout()
    figure.savefig(Path(cfg["output"]) / "training_progress.png", dpi=160)
    plt.show(); plt.close(figure)


## 8. Held-out CAS event evaluation and qualitative inference


In [ ]:
from landslide.evaluate import load_checkpoint, run as evaluate_checkpoint

CAS_EVALUATIONS = {}
for name, cfg in EXPERIMENTS.items():
    checkpoint = Path(cfg["output"]) / "best.pt"
    result = evaluate_checkpoint(
        checkpoint, CAS_DATA_ROOT, split="test", device="cuda",
        data_cfg=cfg["data"], threshold=0.5, amp=True, compute_boundary=True,
        out=Path(cfg["output"]) / "evaluation_cas_test.json",
    )
    CAS_EVALUATIONS[name] = result
    print(name, {key: result[key] for key in ("iou", "dice", "precision", "recall", "f1", "boundary_iou")})
    print("held-out events:", list(result.get("per_event", {})))
save_json(CAS_EVALUATIONS, RUN_ROOT / "cas_test_evaluations.json")


In [ ]:
visual_name = "csg_segformer_b0_cas" if "csg_segformer_b0_cas" in EXPERIMENTS else next(iter(EXPERIMENTS))
visual_cfg = EXPERIMENTS[visual_name]
visual_loader = build_research_dataloader(visual_cfg["data"], "test", batch_size=1, augment=False, num_workers=0, shuffle=False)
visual_batch = next(iter(visual_loader))
visual_model, _, _ = load_checkpoint(Path(visual_cfg["output"]) / "best.pt", torch.device("cuda"))
with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=torch.float16):
    probability = torch.sigmoid(visual_model(visual_batch["image"].to("cuda"))["logits"])[0, 0].float().cpu().numpy()
rgb = visual_batch["image"][0].permute(1, 2, 0).numpy()
rgb = np.clip(rgb * np.asarray(IMAGENET_STD) + np.asarray(IMAGENET_MEAN), 0, 1)
target = visual_batch["mask"][0, 0].numpy()
figure, axes = plt.subplots(1, 4, figsize=(16, 4))
for axis, value, title, cmap in zip(axes, (rgb, target, probability, probability >= 0.5), ("RGB", "Target", "Probability", "Prediction"), (None, "gray", "magma", "gray")):
    axis.imshow(value, cmap=cmap, vmin=0 if cmap else None, vmax=1 if cmap else None); axis.set_title(title); axis.axis("off")
figure.suptitle(f"{visual_name} | {visual_batch['event'][0]}")
figure.tight_layout(); figure.savefig(Path(visual_cfg["output"]) / "held_out_prediction.png", dpi=160)
plt.show(); plt.close(figure)
del visual_model
torch.cuda.empty_cache()


## 9. Optional external HR-GLDD RGB evaluation


In [ ]:
from landslide.evaluate import evaluate_external_hr_gldd

HR_LEDGER = json.loads((COLAB_REPO / "ml/configs/hr_gldd_colab.json").read_text(encoding="utf-8"))
if DOWNLOAD_HR_GLDD_TEST:
    for item in HR_LEDGER["files"]:
        if item["name"] not in {"testX.npy", "testY.npy"}:
            continue
        print(download_once(item["url"], HR_GLDD_ROOT / item["name"], checksum=f"md5:{item['md5']}", expected_bytes=item["bytes"]))
if RUN_HR_GLDD_EVALUATION:
    for required in ("testX.npy", "testY.npy"):
        if not (HR_GLDD_ROOT / required).is_file():
            raise FileNotFoundError(f"Missing HR-GLDD file: {HR_GLDD_ROOT / required}")
    external_results = {}
    for name, cfg in EXPERIMENTS.items():
        external_results[name] = evaluate_external_hr_gldd(
            Path(cfg["output"]) / "best.pt", HR_GLDD_ROOT, split="test",
            device="cuda", amp=True, compute_boundary=True, batch_size=4,
            out=Path(cfg["output"]) / "evaluation_hr_gldd_test.json",
        )
    save_json(external_results, RUN_ROOT / "hr_gldd_evaluations.json")
else:
    print("HR-GLDD evaluation disabled. No external metrics were fabricated.")


## 10. ONNX export, numerical parity, quantization, and benchmarks


In [ ]:
from landslide.deployment import (
    benchmark_onnx, benchmark_torch, export_onnx_with_parity,
    quantize_onnx_dynamic, write_artifact_inventory,
)
from landslide.model import DeploymentModel

DEPLOYMENT_REPORTS = {}
for name, cfg in EXPERIMENTS.items():
    output_dir = Path(cfg["output"])
    checkpoint = output_dir / "best.pt"
    sample_loader = build_research_dataloader(cfg["data"], "test", batch_size=1, augment=False, num_workers=0, shuffle=False)
    sample = next(iter(sample_loader))["image"][:1].contiguous()
    model_path = output_dir / "model.onnx"
    reports = {}
    if RUN_ONNX_EXPORT:
        reports["onnx_export"] = export_onnx_with_parity(
            checkpoint, model_path, image_size=512, sample=sample,
            opset=18, atol=1e-4, rtol=1e-4,
        )
    quantized_path = output_dir / "model.int8.onnx"
    if RUN_QUANTIZATION and model_path.is_file():
        try:
            reports["quantization"] = quantize_onnx_dynamic(
                model_path, quantized_path, sample.numpy(), tolerance=0.05,
            )
        except RuntimeError as exc:
            reports["quantization_error"] = str(exc)
            print(name, "quantization not accepted:", exc)
    if RUN_BENCHMARKS:
        torch_model, _, _ = load_checkpoint(checkpoint, torch.device("cuda"))
        reports["pytorch_benchmark"] = benchmark_torch(
            DeploymentModel(torch_model), sample, device="cuda",
            warmup=5, repetitions=BENCHMARK_REPETITIONS,
        )
        del torch_model
        torch.cuda.empty_cache()
        if model_path.is_file():
            reports["onnx_cpu_benchmark"] = benchmark_onnx(
                model_path, sample.numpy(), warmup=5, repetitions=BENCHMARK_REPETITIONS,
            )
        quantization = reports.get("quantization", {})
        if quantized_path.is_file() and quantization.get("artifact_usable"):
            reports["onnx_int8_cpu_benchmark"] = benchmark_onnx(
                quantized_path, sample.numpy(), warmup=5, repetitions=BENCHMARK_REPETITIONS,
            )
    save_json(reports, output_dir / "deployment_report.json")
    DEPLOYMENT_REPORTS[name] = reports
save_json(DEPLOYMENT_REPORTS, RUN_ROOT / "deployment_reports.json")


## 11. Persist and checksum all artifacts


In [ ]:
inventories = {}
for name, cfg in EXPERIMENTS.items():
    inventories[name] = write_artifact_inventory(Path(cfg["output"]), "artifact_manifest.json")
save_json({"git_sha": GIT_SHA, "experiments": inventories}, RUN_ROOT / "artifact_index.json")
print("Persistent run directory:", RUN_ROOT)
for name, report in inventories.items():
    print(name, "files:", report["file_count"])


## Completion criteria

A completed run has a resolved config, runtime report, `best.pt`, `last.pt`, history JSON/CSV, training plot, held-out CAS metrics with per-event results, qualitative prediction, ONNX parity report, supported quantization report, benchmark report, and SHA-256 artifact manifest under the persistent Drive run directory. Empty or interrupted runs are not scientific results; rerun the training cell with the same `RUN_NAME` to resume.
